In [1]:
# work environment: faiss_env
import pandas as pd
import numpy as np
import pickle
import sys
import os
import warnings
from pathlib import Path
from scipy.spatial import KDTree

# 경로 설정
gems_tco_path = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
sys.path.append(gems_tco_path)

# Warnings 무시 설정
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="xarray")

# 커스텀 모듈 로드
from GEMS_TCO import configuration as config
from GEMS_TCO import data_preprocess as dmbh

# =====================================================================
# [수정된 핵심 모듈] 다이내믹 격자 매칭 및 마스킹(Penalty) 함수
# =====================================================================
def dynamic_match_for_whittle(loaded_map, base_center_points, step_lat, step_lon, v_drift_lon=-0.0048):
    """
    시간대별로 격자를 이동(Dynamic Shift)시키며 KDTree로 매칭하고,
    거리가 멀면(Penalty) NaN으로 처리한 뒤, DW 모델을 위해 고정된 좌표계로 반환합니다.
    """
    coarse_cen_map = {}
    sorted_keys = sorted(loaded_map.keys())

    # Penalty 기준: 격자 간격의 정확히 절반
    lat_thresh = step_lat / 2.0
    lon_thresh = step_lon / 2.0
    
    # 💥 에러 픽스: DataFrame으로 들어올 경우 컬럼 이름에 의존하지 않고 무조건 첫 두 열을 가져옴!
    if isinstance(base_center_points, pd.DataFrame):
        base_center_points = base_center_points.iloc[:, 0:2].values
    elif isinstance(base_center_points, list):
        base_center_points = np.array(base_center_points)

    for i, key in enumerate(sorted_keys):
        t_idx = i % 8  # 하루 8시간 기준 인덱스 (0~7)
        df_raw = loaded_map[key]

        if len(df_raw) == 0:
            continue

        # 1. 시간에 따른 격자 서쪽 이동 (Dynamic Shift)
        shift_lat = 0.0
        shift_lon = t_idx * v_drift_lon
        shifted_grid = base_center_points + np.array([shift_lat, shift_lon])

        # 💥 에러 픽스 2: 원본 데이터(orbit_map)도 컬럼 이름이 다를 수 있으므로 안전하게 추출
        if 'Latitude' in df_raw.columns and 'Longitude' in df_raw.columns:
            raw_coords = df_raw[['Latitude', 'Longitude']].values
        else:
            # 이름이 다르면 첫 번째, 두 번째 열을 위경도로 간주
            raw_coords = df_raw.iloc[:, 0:2].values

        # 2. KDTree를 이용한 매칭
        tree = KDTree(raw_coords)
        distances, indices = tree.query(shifted_grid)

        # 3. 매칭된 데이터 추출 및 페널티(탈선 마스킹) 계산
        matched_raw_coords = raw_coords[indices]
        lat_diffs = np.abs(shifted_grid[:, 0] - matched_raw_coords[:, 0])
        lon_diffs = np.abs(shifted_grid[:, 1] - matched_raw_coords[:, 1])
        
        # 거리가 절반을 넘어가면 탈선(Invalid)으로 간주
        invalid_mask = (lat_diffs > lat_thresh) | (lon_diffs > lon_thresh)

        # 4. DataFrame 조립
        df_matched = df_raw.iloc[indices].copy().reset_index(drop=True)

        # 거리가 너무 먼 데이터(탈선)는 일괄 NaN 처리
        for col in df_matched.columns:
            df_matched.loc[invalid_mask, col] = np.nan

        # 5. 모델을 속이기 위해 위치는 '고정된 원래 격자(Base Grid)'로 덮어쓰기
        df_matched['Latitude'] = base_center_points[:, 0]
        df_matched['Longitude'] = base_center_points[:, 1]

        # 디버깅 및 원래 위치 보존을 위해 실제 위성 좌표 기록
        df_matched['Source_Latitude'] = matched_raw_coords[:, 0]
        df_matched['Source_Longitude'] = matched_raw_coords[:, 1]
        df_matched.loc[invalid_mask, 'Source_Latitude'] = np.nan
        df_matched.loc[invalid_mask, 'Source_Longitude'] = np.nan

        coarse_cen_map[key] = df_matched

    return coarse_cen_map
# =====================================================================

def process_coarse_data(base_path, years, months, lat_lon_bounds, step_sizes, base_csv_path, grid_type='standard'):
    """
    격자화된 데이터를 생성하고 저장하는 통합 함수
    """
    print(f"\n--- Starting Coarsening Process for Whittle (Type: {grid_type}) ---")
    
    lat_start, lat_end, lon_start, lon_end = lat_lon_bounds
    step_lat, step_lon = step_sizes
    
    # 1. Base DataFrame & Instance 초기화
    try:
        print(f"Loading base dataframe from: {base_csv_path}")
        df = pd.read_csv(base_csv_path)
        instance = dmbh.center_matching_hour(df, lat_start, lat_end, lon_start, lon_end)
    except Exception as e:
        print(f"Error initializing instance: {e}")
        return

    # 2. 격자 포인트(Base Grid) 생성
    if grid_type == 'rect':
        print("Generating Rectangular Center Points (No Calibration)...")
        center_points = instance.make_center_points_wo_calibration(step_lat=step_lat, step_lon=step_lon)
        file_suffix = "rect" 
    else:
        print("Generating Standard Center Points (With Calibration)...")
        center_points = instance.make_center_points(step_lat=step_lat, step_lon=step_lon)
        file_suffix = "std"

    # 3. 연도/월별 처리 루프
    for year in years:
        for month in months:
            month_str = f"{month:02d}"
            print(f">> Processing: Year {year}, Month {month_str}")
            
            try:
                pickle_path = os.path.join(base_path, f'pickle_{year}')
                input_filename = f"orbit_map{str(year)[2:]}_{month_str}.pkl"
                
                if file_suffix == "std":
                    output_filename = f"coarse_cen_map_whittle_{str(year)[2:]}_{month_str}.pkl"
                else:
                    output_filename = f"coarse_cen_map_{file_suffix}_whittle_{str(year)[2:]}_{month_str}.pkl"
                
                input_filepath = os.path.join(pickle_path, input_filename)
                output_filepath = os.path.join(pickle_path, output_filename)
                
                # Load Original Orbit Data
                if not os.path.exists(input_filepath):
                    print(f"   [Skip] File not found: {input_filename}")
                    continue

                with open(input_filepath, 'rb') as pickle_file:
                    loaded_map = pickle.load(pickle_file)

                # -------------------------------------------------------------
                # 💥 다이내믹 격자 매칭 함수 적용!
                # -------------------------------------------------------------
                coarse_cen_map = dynamic_match_for_whittle(
                    loaded_map=loaded_map, 
                    base_center_points=center_points, 
                    step_lat=step_lat, 
                    step_lon=step_lon
                )

                # Save Processed Data
                os.makedirs(pickle_path, exist_ok=True)
                with open(output_filepath, 'wb') as pickle_file:
                    pickle.dump(coarse_cen_map, pickle_file)
                
                print(f"   [Saved] {output_filename}")

            except Exception as e:
                print(f"   [Error] {e}")

# ==========================================
# 실행부 (Main)
# ==========================================
if __name__ == '__main__':
    BASE_PATH = config.mac_data_load_path 
    
    target_year = 2024
    target_month = 7
    
    LAT_LON_BOUNDS = (-3, 2, 121, 131)
    STEP_SIZES = (0.044, 0.063)

    BASE_CSV_PATH = f"/Users/joonwonlee/Documents/GEMS_DATA/data_{target_year}/data_{str(target_year)[2:]}_07_0131_N-32_E121131.csv"
    YEARS_TO_PROCESS = [target_year]
    MONTHS_TO_PROCESS = [target_month]
    
    # 직사각형 격자(rect)에 다이내믹 매칭을 적용하여 피클 저장
    process_coarse_data(
        base_path=BASE_PATH,
        years=YEARS_TO_PROCESS,
        months=MONTHS_TO_PROCESS,
        lat_lon_bounds=LAT_LON_BOUNDS,
        step_sizes=STEP_SIZES,
        base_csv_path=BASE_CSV_PATH,
        grid_type='rect'  
    )


--- Starting Coarsening Process for Whittle (Type: rect) ---
Loading base dataframe from: /Users/joonwonlee/Documents/GEMS_DATA/data_2024/data_24_07_0131_N-32_E121131.csv
Generating Rectangular Center Points (No Calibration)...
>> Processing: Year 2024, Month 07
   [Saved] coarse_cen_map_rect_whittle_24_07.pkl


아래는 위에서 저장 제대로 했는지 테스팅

In [2]:
import sys
# Add your custom path
gems_tco_path = "/Users/joonwonlee/Documents/GEMS_TCO-1/src"
sys.path.append(gems_tco_path)
from GEMS_TCO import kernels_reparam_space_time_gpu as kernels_reparam_space_time
from GEMS_TCO import orderings as _orderings 
from GEMS_TCO import alg_optimization, BaseLogger

from typing import Optional, List, Tuple
from pathlib import Path
import typer
import json
from json import JSONEncoder
from GEMS_TCO import configuration as config
from GEMS_TCO.data_loader import load_data2, exact_location_filter
from GEMS_TCO.data_loader import load_data_dynamic_processed # 새 클래스 임포트
from GEMS_TCO import debiased_whittle
from torch.nn import Parameter

space: List[str] = ['1', '1']
lat_lon_resolution = [int(s) for s in space]
mm_cond_number: int = 8


#years = ['2024']
years = ['2024']
month_range = [7] 

output_path = input_path = Path(config.mac_estimates_day_path)
data_load_instance = load_data_dynamic_processed(config.mac_data_load_path)

#lat_range_input = [1, 3]
#lon_range_input = [125.0, 129.0]

lat_range_input=[-3,2]      
lon_range_input=[121, 131.0] 

df_map, ord_mm, nns_map, day_offsets = data_load_instance.load_maxmin_ordered_data_bymonthyear(
lat_lon_resolution=lat_lon_resolution, 
mm_cond_number=mm_cond_number,
years_=years, 
months_=month_range,

lat_range=lat_range_input,   
lon_range=lon_range_input,
is_whittle= True
)

import torch

--- Global Monthly Mean for 2024-7: 257.9726 ---


In [3]:
daily_aggregated_tensors_dw = [] 
daily_hourly_maps_dw = []      

daily_aggregated_tensors_vecc = [] 
daily_hourly_maps_vecc = []   

for day_index in range(31):
    hour_start_index = day_index * 8
    hour_end_index = (day_index + 1) * 8
    hour_indices = [hour_start_index, hour_end_index]

    # --- DW용 데이터 로드 (day_offsets 인자 추가) ---
    day_hourly_map, day_aggregated_tensor = data_load_instance.load_working_data(
        df_map, 
        day_offsets,  # <--- 이 부분이 추가되어야 합니다
        hour_indices, 
        ord_mm=None,
        dtype=torch.float64, 
        keep_ori=False
    )
    daily_aggregated_tensors_dw.append(day_aggregated_tensor)
    daily_hourly_maps_dw.append(day_hourly_map)

    # --- Vecchia용 데이터 로드 (day_offsets 인자 추가) ---
    day_hourly_map, day_aggregated_tensor = data_load_instance.load_working_data(
        df_map, 
        day_offsets,  # <--- 이 부분이 추가되어야 합니다
        hour_indices, 
        ord_mm=ord_mm,
        dtype=torch.float64, 
        keep_ori= True
    )
    daily_aggregated_tensors_vecc.append(day_aggregated_tensor)
    daily_hourly_maps_vecc.append(day_hourly_map)

print(f"Aggregated Tensor Shape: {daily_aggregated_tensors_vecc[0].shape}")
# 예상 출력: torch.Size([행수, 12]) -> 열이 12개여야 성공입니다.
nn = daily_aggregated_tensors_vecc[0].shape[0]

Aggregated Tensor Shape: torch.Size([145008, 11])


In [6]:
print(daily_hourly_maps_dw[2]['2024_07_y24m07day03_hm00:53'].shape)

daily_hourly_maps_dw[2]['2024_07_y24m07day03_hm00:53']

torch.Size([18126, 11])


tensor([[  2.0000, 131.0000,      nan,  ...,   0.0000,   0.0000,   0.0000],
        [  2.0000, 130.9370,   8.2864,  ...,   0.0000,   0.0000,   0.0000],
        [  2.0000, 130.8740,   4.8700,  ...,   0.0000,   0.0000,   0.0000],
        ...,
        [ -2.9720, 121.1720,   1.6025,  ...,   0.0000,   0.0000,   0.0000],
        [ -2.9720, 121.1090,  -1.2228,  ...,   0.0000,   0.0000,   0.0000],
        [ -2.9720, 121.0460,  -2.2965,  ...,   0.0000,   0.0000,   0.0000]],
       dtype=torch.float64)